In [ ]:
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, make_scorer, precision_score, recall_score
from sklearn.model_selection import GridSearchCV
from import_ipynb import NotebookFinder  # type: ignore
from IPython.display import display, Markdown, HTML
import importlib
from colorama import Fore, Style
import joblib
import os
import pandas as pd

In [ ]:
# retrouver les dossiers
root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
visualisation_dir = os.path.join(root, "pneumonia_knn", "notebooks", "visualisation")
hyperparameter_dir = os.path.join(root, "pneumonia_knn", "documents", "hyperparameter")

# charger les fichiers
# --- pneumonia_knn\notebooks\visualisation\accuracy_metrics.ipynb
spec_metrics = NotebookFinder().find_spec("metrics", [visualisation_dir])
metrics = importlib.util.module_from_spec(spec_metrics)
spec_metrics.loader.exec_module(metrics)

In [ ]:
def train_knn_pca(grid, X_test_scaled, y_test):
    knn_pca = grid.best_estimator_
    y_pred_pca = knn_pca.predict(X_test_scaled)

    metrics_results = pd.DataFrame(grid.cv_results_)

    metrics.show_scores(metrics_results, 'Résultats GridSearchCV - KNN + PCA')
    print(f"{Fore.YELLOW} Note : \n recall = combien de malade l'IA a réussi à detecter en tant que malades. \n précison = parmis ce que l'IA a detecté lesquels sont vraiment malades.\n auc = La prédiction est mesurée sur 1 (1 étant l'idéa).{Style.RESET_ALL}")
    accuracy_with_pca = accuracy_score(y_test, y_pred_pca)
    return accuracy_with_pca

In [ ]:
# KNN avec PCA
def implementation_with_PCA(X_train_scaled,X_test_scaled, y_train, y_test):
    if os.path.exists(f'{hyperparameter_dir}/knn_pca_grid_search.pkl'):
        grid = joblib.load(f'{hyperparameter_dir}/knn_pca_grid_search.pkl')
        train_knn_pca(grid, X_test_scaled, y_test)
    else:
        pipeline = Pipeline([
            ('pca', PCA()),
            ('knn', KNeighborsClassifier())
        ])
        # 2. Définir les hyperparamètres
        param_grid = {
            'pca__n_components': [50, 100],
            'knn__n_neighbors': [3, 5, 9],
            'knn__metric': ['euclidean', 'manhattan'],
            'knn__weights': ['distance'],
            'knn__algorithm': ['ball_tree']
        }
        scoring = {
            'accuracy':  'accuracy',
            'precision': make_scorer(precision_score, average='weighted'),
            'recall':    make_scorer(recall_score,    average='weighted'),
        }    # 3. Lancer GridSearchCV
        grid = GridSearchCV(
            estimator=pipeline,
            param_grid=param_grid,
            cv=2,
            scoring=scoring,
            refit='accuracy',
            n_jobs=1,
            verbose=0  # affiche ou non l'avancement
        )
        grid.fit(X_train_scaled, y_train)
        joblib.dump(grid, f'{hyperparameter_dir}/knn_pca_grid_search.pkl')
        train_knn_pca(grid, X_test_scaled, y_test)